# طراحی معکوس رینگ رزوناتور — گام ۱: ناحیهٔ کوپلینگ باس–رینگ (Meep adjoint)

به‌جای طراحی کل رینگ (که Q بالا و شبیه‌سازی بسیار طولانی دارد)، فقط **ناحیهٔ کوپلینگ** را طراحی معکوس می‌کنیم:
یک بلوک ۴ پورتی کوچک (باس + موجبر رینگ). هر تکرار adjoint در حد چند ثانیه تا دقیقه است.

**مراحل**
1. از مشخصات هدف (`Q_TARGET`) و افت چرخه، مقدار لازم `κ²` (توان تزویج باس→رینگ) را از مدل add-drop حل می‌کنیم.
2. کوپلر ۴ پورتی را طوری بهینه می‌کنیم که `κ²` روی **کل باند ۱٫۵–۱٫۶ µm تقریباً ثابت** بماند، با بازتاب و تزویج پس‌رو ناچیز.
3. دو نمونه (پایین: باس ورودی/Through، بالا: باس Drop) را در یک رینگ مسابقه‌ای (racetrack) مونتاژ و با FDTD راستی‌آزمایی می‌کنیم.

> اول `QUICK_TEST = True` را امتحان کنید. مقدار `A_RT_DB` را از خروجی نوت‌بوک رینگ معمولی خودتان جایگزین کنید.

In [ ]:
!pip install -q condacolab
import condacolab
condacolab.install()

In [ ]:
!mamba install -y -q -c conda-forge pymeep autograd scipy matplotlib
!pip install -q gdstk

## ۰) پارامترها

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import meep as mp
import meep.adjoint as mpa
from autograd import numpy as npa
from autograd import tensor_jacobian_product
from scipy.signal import find_peaks, peak_widths
from scipy.optimize import least_squares

try:
    mp.verbosity(0)
except Exception:
    pass

# ======================= switches =======================
QUICK_TEST = False
GRAD_CHECK = False

RES_OPT = 10 if QUICK_TEST else 20       # px/um during optimisation
RES_VER = 16 if QUICK_TEST else 32       # px/um for verification
NF_OPT = 3 if QUICK_TEST else 5          # wavelengths in the optimisation
NF_CPL = 11 if QUICK_TEST else 31        # wavelengths for coupler verification
NF_RING = 101 if QUICK_TEST else 401     # wavelengths for the ring spectrum
PARITY = mp.EVEN_Z

# ======================= technology (same as before) =======================
t_si, n_clad, n_si = 0.22, 1.444, 3.476


def slab_neff(t, n1=n_si, n2=n_clad, lam0=1.55):
    k0 = 2 * np.pi / lam0
    a = k0 * t / 2
    V = a * np.sqrt(n1 ** 2 - n2 ** 2)
    lo, hi = 1e-9, min(V, np.pi / 2 - 1e-9)
    f = lambda u: u * np.tan(u) - np.sqrt(max(V ** 2 - u ** 2, 0.0))
    for _ in range(200):
        mid = 0.5 * (lo + hi)
        if f(mid) > 0:
            hi = mid
        else:
            lo = mid
    u = 0.5 * (lo + hi)
    return float(np.sqrt(n1 ** 2 - (u / a) ** 2))


n_core = slab_neff(t_si)
Si2D = mp.Medium(index=n_core)
SiO2 = mp.Medium(index=n_clad)
w = 0.45
dpml = 1.0
NG_EST = 3.3                              # only for choosing run times

# ======================= bands =======================
# coupler is optimised/verified over the wide band (flat kappa^2), the ring is simulated in a narrow window
C_WL_MIN, C_WL_MAX = 1.50, 1.60
cmin, cmax = 1 / C_WL_MAX, 1 / C_WL_MIN
cmid, cspan = 0.5 * (cmin + cmax), cmax - cmin
R_WL_MIN, R_WL_MAX = 1.53, 1.57
rmin, rmax = 1 / R_WL_MAX, 1 / R_WL_MIN
rmid, rspan = 0.5 * (rmin + rmax), rmax - rmin
fcen_src, fwidth_src = 1 / 1.55, 0.12
src_size = 1.2                            # ports are close: monitors must not overlap

# ======================= ring (racetrack) layout =======================
PORT_SEP = 1.2                            # centre-to-centre distance of bus and ring waveguides at the DR edges
hs = PORT_SEP / 2
R_LOOP = 5.0                              # radius of the two semicircles (um) - same bend radius as the add-drop ring you simulated
L_ST = 0.5                                # straight stubs outside each design region (um)

# ======================= inverse-design region =======================
DR_X, DR_Y = 5.0, 3.2
PORT_STUB = 0.4
FILTER_R = 0.15
ETA = 0.5
W_REFL = 0.5

Nx = int(DR_X * RES_OPT) + 1              # conic_filter grid: L*res + 1 points per axis
Ny = int(DR_Y * RES_OPT) + 1
assert Ny % 2 == 1, "DR_Y*RES_OPT must be an even integer"
Nyh = (Ny + 1) // 2

L_RING_EST = 2 * np.pi * R_LOOP + 2 * (DR_X + 2 * L_ST)
print(f"n_core = {n_core:.4f},  estimated ring length = {L_RING_EST:.1f} um")

## ۱) مشخصات هدف → `κ²` لازم

برای رینگ add-drop متقارن (`r² = 1 − κ²`، دامنهٔ عبور چرخه `a`):
`FWHM ≈ (1 − r²a)·λ² / (π·n_g·L·r·√a)` و `Q = λ / FWHM`.
`A_RT_DB` افت چرخهٔ فیت‌شدهٔ رینگ معمولی شماست (۰٫۷۸ dB، شامل تشعشع خم و افت کوپلر). با این افت، سقف Q حدود ۳۰۰۰ است و
Q بالاتر فقط با کاهش افت (مثلاً شعاع بزرگ‌تر یا خم طراحی‌معکوس در گام ۲) ممکن است. جدول بده‌بستان `κ²` در خروجی چاپ می‌شود.

In [ ]:
# ======================= TARGET SPECIFICATION (edit me) =======================
Q_TARGET = 1500          # desired loaded Q (with ~0.78 dB round-trip loss the ceiling is only ~3000, see table below)
A_RT_DB = 0.78           # round-trip loss in dB, from your add-drop fit (a = 0.9146); it includes bend radiation in 2D
KAPPA2_OVERRIDE = None   # set e.g. 0.03 to skip the Q-based choice


def neff_curve(freq_list, width, n_core_, res):
    sim = mp.Simulation(cell_size=mp.Vector3(1, 5, 0),
                        geometry=[mp.Block(size=mp.Vector3(mp.inf, width, mp.inf), center=mp.Vector3(),
                                           material=mp.Medium(index=n_core_))],
                        default_material=SiO2, resolution=res, dimensions=2)
    sim.init_sim()
    vol_ = mp.Volume(center=mp.Vector3(), size=mp.Vector3(0, 4, 0))
    return np.array([sim.get_eigenmode(f, mp.X, vol_, 1, mp.Vector3(2.3 * f, 0, 0), parity=PARITY).k.x / f
                     for f in freq_list])


f_sub = np.linspace(rmin, rmax, 5)
neff_sub = neff_curve(f_sub, w, n_core, RES_VER)
lam_sub = 1 / f_sub
ng_all = neff_sub - lam_sub * np.gradient(neff_sub, lam_sub)
ng_c = float(np.interp(rmid, f_sub, ng_all))
lam0 = 1 / rmid
print(f"n_eff = {np.interp(rmid, f_sub, neff_sub):.4f},  n_g = {ng_c:.3f}")


def ring_metrics(K, a_dB, ng, L, lam_):
    r = np.sqrt(1 - K)
    a = 10 ** (-a_dB / 20)
    D0 = (1 - r * r * a) ** 2
    Td = (1 - r * r) ** 2 * a / D0               # lossless couplers assumed in this design table
    Tmin = r * r * (1 - a) ** 2 / D0
    fwhm = (1 - r * r * a) * lam_ ** 2 / (np.pi * ng * L * r * np.sqrt(a))
    return lam_ / fwhm, Td, Tmin, fwhm


Ks = np.logspace(-3, np.log10(0.5), 400)
Qs = np.array([ring_metrics(K, A_RT_DB, ng_c, L_RING_EST, lam0)[0] for K in Ks])
if KAPPA2_OVERRIDE is None:
    KAPPA2 = float(Ks[np.argmin(np.abs(Qs - Q_TARGET))])
else:
    KAPPA2 = float(KAPPA2_OVERRIDE)
Qd, Tdp, Tmn, fw = ring_metrics(KAPPA2, A_RT_DB, ng_c, L_RING_EST, lam0)
print("kappa^2 trade-off for this loss (model):")
for K_ in (0.01, 0.03, 0.05, 0.08, 0.12, 0.19, 0.25):
    Q_, Td_, Tm_, _ = ring_metrics(K_, A_RT_DB, ng_c, L_RING_EST, lam0)
    print(f"   kappa^2 = {K_:.3f} | Q = {Q_:6.0f} | drop peak = {10*np.log10(Td_):6.2f} dB | through min = {10*np.log10(max(Tm_, 1e-12)):6.1f} dB")
print(f"==> required coupler  kappa^2 = {KAPPA2:.4f}")
print(f"    expected: Q_loaded = {Qd:,.0f}, FWHM = {fw*1e3:.3f} nm, drop peak = {10*np.log10(Tdp):.2f} dB, "
      f"through min = {10*np.log10(max(Tmn,1e-12)):.1f} dB")
if Qs.max() < Q_TARGET:
    print("Warning: Q_TARGET is above what this loss allows; kappa^2 hits the lower search limit.")

plt.figure(figsize=(6, 3.5))
plt.loglog(Ks, Qs); plt.axvline(KAPPA2, color="r", ls="--"); plt.axhline(Q_TARGET, color="gray", ls=":")
plt.xlabel("coupler kappa^2"); plt.ylabel("loaded Q (model)"); plt.grid(alpha=0.3, which="both"); plt.show()

## ۲) طراحی معکوس کوپلر ۴ پورتی

پورت‌ها (مرکز ناحیه در مبدأ): باس در `y = −hs`، موجبر رینگ در `y = +hs`.
`P0` = باس چپ (ورودی)، `Through` = باس راست، `Cross` = رینگ راست (همان `κ²`)، `Back` = رینگ چپ (تزویج پس‌رو، باید صفر شود).
تقارن آینه‌ای نسبت به `y = 0` اعمال می‌شود (کوپلر متقارن → اختلاف فاز ۹۰° بین عبور و تزویج).

In [ ]:
xs_pix = np.linspace(-DR_X / 2, DR_X / 2, Nx)
ys_pix = np.linspace(-DR_Y / 2, DR_Y / 2, Ny)
MASK = (((xs_pix < -DR_X / 2 + PORT_STUB) | (xs_pix > DR_X / 2 - PORT_STUB))[:, None]
        & (np.abs(np.abs(ys_pix) - hs) <= w / 2)[None, :]).astype(float)


def mapping(x, eta, beta):
    half = npa.reshape(x, (Nx, Nyh))
    full = npa.concatenate([half, half[:, -2::-1]], axis=1)      # mirror about the centre row (y = 0)
    full = full * (1 - MASK) + MASK                              # keep the four ports connected
    filt = mpa.conic_filter(npa.ravel(full), FILTER_R, DR_X, DR_Y, RES_OPT)
    proj = mpa.tanh_projection(filt, beta, eta)
    return npa.ravel(proj)


# ---- single-coupler simulation cell
wg_len = 2.5
sxc = DR_X + 2 * (wg_len + dpml)
syc = DR_Y + 2 * dpml + 1.0
XS_C, XMI_C, XMO_C = -DR_X / 2 - 1.8, -DR_X / 2 - 1.2, DR_X / 2 + 1.5
T_C = NG_EST * sxc + 40


def stub_blocks():
    xl, xr = -DR_X / 2 + 0.3, DR_X / 2 - 0.3
    out = []
    for yc in (-hs, +hs):
        out.append(mp.Block(size=mp.Vector3(xl + sxc, w, mp.inf), center=mp.Vector3(0.5 * (-sxc + xl), yc, 0), material=Si2D))
        out.append(mp.Block(size=mp.Vector3(sxc - xr, w, mp.inf), center=mp.Vector3(0.5 * (xr + sxc), yc, 0), material=Si2D))
    return out


def make_csim(geometry, res):
    src = mp.EigenModeSource(mp.GaussianSource(fcen_src, fwidth=fwidth_src), center=mp.Vector3(XS_C, -hs, 0),
                             size=mp.Vector3(0, src_size, 0), eig_band=1, eig_parity=PARITY, eig_match_freq=True)
    return mp.Simulation(cell_size=mp.Vector3(sxc, syc, 0), boundary_layers=[mp.PML(dpml)], geometry=geometry,
                         sources=[src], default_material=SiO2, resolution=res, dimensions=2)


def cvol(x, y):
    return mp.Volume(center=mp.Vector3(x, y, 0), size=mp.Vector3(0, src_size, 0))


def cref_alpha(res, nf):
    sim = make_csim([mp.Block(size=mp.Vector3(mp.inf, w, mp.inf), center=mp.Vector3(0, -hs, 0), material=Si2D)], res)
    mon = sim.add_mode_monitor(cmid, cspan, nf, mp.ModeRegion(center=mp.Vector3(XMO_C, -hs, 0), size=mp.Vector3(0, src_size, 0)))
    sim.run(until_after_sources=T_C)
    return np.array(sim.get_eigenmode_coefficients(mon, [1], eig_parity=PARITY).alpha[0])[:, 0], np.array(mp.get_flux_freqs(mon))


print("Reference run for optimisation normalisation ...")
a_ref_opt, freqs_opt = cref_alpha(RES_OPT, NF_OPT)
ref_pow = np.abs(a_ref_opt) ** 2
print("optimisation wavelengths (um):", np.round(1 / freqs_opt, 4))

In [ ]:
design_variables = mp.MaterialGrid(mp.Vector3(Nx, Ny), SiO2, Si2D, grid_type="U_MEAN")
design_region = mpa.DesignRegion(design_variables, volume=mp.Volume(center=mp.Vector3(), size=mp.Vector3(DR_X, DR_Y, 0)))
geo_opt = stub_blocks() + [mp.Block(center=mp.Vector3(), size=mp.Vector3(DR_X, DR_Y, 0), material=design_variables)]
sim_opt = make_csim(geo_opt, RES_OPT)

ob_thru = mpa.EigenmodeCoefficient(sim_opt, cvol(XMO_C, -hs), 1, eig_parity=PARITY)                  # bus  -> bus right
ob_cross = mpa.EigenmodeCoefficient(sim_opt, cvol(XMO_C, +hs), 1, eig_parity=PARITY)                 # bus  -> ring right (kappa^2)
ob_refl = mpa.EigenmodeCoefficient(sim_opt, cvol(XMI_C, -hs), 1, forward=False, eig_parity=PARITY)   # reflection
ob_back = mpa.EigenmodeCoefficient(sim_opt, cvol(XMI_C, +hs), 1, forward=False, eig_parity=PARITY)   # ring left (backward)

K_T = KAPPA2
S_SC = max(K_T, 0.05)                                         # error scale (relative for small kappa^2)


W_CROSS = 1.0


def J(a_t, a_k, a_r, a_b):
    Pt = npa.abs(a_t) ** 2 / ref_pow
    Pk = npa.abs(a_k) ** 2 / ref_pow
    Pr = npa.abs(a_r) ** 2 / ref_pow
    Pb = npa.abs(a_b) ** 2 / ref_pow
    return -npa.mean(W_CROSS * ((Pk - K_T) / S_SC) ** 2 + ((Pt - (1 - K_T)) / S_SC) ** 2 + W_REFL * (Pr + Pb) / S_SC)


opt = mpa.OptimizationProblem(simulation=sim_opt, objective_functions=[J],
                              objective_arguments=[ob_thru, ob_cross, ob_refl, ob_back],
                              design_regions=[design_region], frequencies=freqs_opt,
                              decay_by=1e-5, maximum_run_time=250)


def reduce_grad(dJ, n):
    dJ = np.squeeze(np.asarray(dJ, dtype=float))
    if dJ.ndim == 2:
        dJ = dJ.sum(axis=1) if dJ.shape[0] == n else dJ.sum(axis=0)
    assert dJ.size == n, f"unexpected gradient shape {dJ.shape}"
    return dJ


LAST = {"Pk": float("nan"), "Pt": float("nan")}


def fom_and_grad(x, beta):
    f0, dJ = opt([mapping(x, ETA, beta)])
    try:                                              # progress diagnostics (mean cross / through power)
        rl = opt.results_list
        LAST["Pk"] = float(np.mean(np.abs(np.asarray(rl[1])) ** 2 / ref_pow))
        LAST["Pt"] = float(np.mean(np.abs(np.asarray(rl[0])) ** 2 / ref_pow))
    except Exception:
        pass
    g = tensor_jacobian_product(mapping, 0)(x, ETA, beta, reduce_grad(dJ, Nx * Ny))
    return float(np.real(np.squeeze(f0))), np.asarray(g)


rng = np.random.default_rng(0)
# Starting design: bus straight at y=-hs, ring waveguide at y=+hs, both bending towards each other in the middle
# (a short directional-coupler-like section). Starting from uniform gray gets stuck in the "straight bus" optimum.
G0, SIG = 0.15, 1.2                                           # initial minimum gap (um) and width of the dip (um)
d_c = (hs - (w / 2 + G0 / 2)) * np.exp(-(xs_pix / SIG) ** 2)
yc_low = -(hs - d_c)
rho0 = np.zeros((Nx, Ny))
for i in range(Nx):
    rho0[i, :] = ((np.abs(ys_pix - yc_low[i]) <= w / 2) | (np.abs(ys_pix + yc_low[i]) <= w / 2)).astype(float)
x0 = np.clip(rho0[:, :Nyh].ravel() + 0.02 * rng.standard_normal(Nx * Nyh), 0, 1)

f_i, _ = fom_and_grad(x0, 4.0)
print(f"start design: J = {f_i:.4f} | mean cross = {LAST['Pk']:.4f} (target {K_T:.4f}) | mean through = {LAST['Pt']:.4f}")
print("-> if the start cross power is far above/below the target, decrease/increase G0 (smaller gap = stronger coupling)")

if GRAD_CHECK:
    beta_t = 4.0
    f_c, g_c = fom_and_grad(x0, beta_t)
    for i in rng.choice(Nx * Nyh, 3, replace=False):
        d = 0.02
        xp, xm = x0.copy(), x0.copy(); xp[i] += d; xm[i] -= d
        fp, _ = fom_and_grad(xp, beta_t); fm, _ = fom_and_grad(xm, beta_t)
        print(f"idx {i}: adjoint = {g_c[i]: .3e}   finite-diff = {(fp - fm) / (2 * d): .3e}")

In [ ]:
BETAS = [4, 8, 16, 32, 64]
N_IT = [2] * len(BETAS) if QUICK_TEST else [20] * len(BETAS)
LR = 0.04

x = x0.copy(); m = np.zeros_like(x); v = np.zeros_like(x); step = 0
J_hist, beta_hist = [], []
t_start = time.time()
for beta, nit in zip(BETAS, N_IT):
    for it in range(nit):
        t0 = time.time()
        f0, g = fom_and_grad(x, beta)
        step += 1
        m = 0.9 * m + 0.1 * g
        v = 0.999 * v + 0.001 * g * g
        x = np.clip(x + LR * (m / (1 - 0.9 ** step)) / (np.sqrt(v / (1 - 0.999 ** step)) + 1e-8), 0, 1)
        J_hist.append(f0); beta_hist.append(beta)
        print(f"beta={beta:>3}  it={it + 1:>2}/{nit}  J={f0: .5f}  cross={LAST['Pk']:.4f} (target {K_T:.4f})  "
              f"through={LAST['Pt']:.4f}  ({time.time() - t0:.0f} s)")
print(f"Optimisation finished in {(time.time() - t_start) / 60:.1f} min")

plt.figure(figsize=(6, 3.5))
plt.plot(J_hist, "o-", ms=3)
for i in np.where(np.diff(beta_hist) != 0)[0]:
    plt.axvline(i + 0.5, color="gray", ls=":", lw=0.8)
plt.xlabel("iteration"); plt.ylabel("J (0 = ideal)"); plt.grid(alpha=0.3); plt.title("Convergence"); plt.show()

np.save("ringcoupler_design_x.npy", x)
rho_final = np.asarray(mapping(x, ETA, 256.0)).reshape(Nx, Ny)
np.save("ringcoupler_design_rho.npy", rho_final)

plt.figure(figsize=(7, 4.2))
plt.imshow(rho_final.T, origin="lower", cmap="gray_r", extent=[-DR_X / 2, DR_X / 2, -DR_Y / 2, DR_Y / 2])
plt.xlabel("x (um)"); plt.ylabel("y (um)"); plt.title("Inverse-designed bus-ring coupler (black = Si)"); plt.show()

### (اختیاری) ادامهٔ بهینه‌سازی از طرح ذخیره‌شده

اگر `κ²` میانگین زیر هدف مانده یا با طول‌موج می‌لغزد: `REFINE = True`. وزن خطای تزویج بیشتر می‌شود و از همان طرح ادامه می‌دهد.

In [ ]:
REFINE = False
REFINE_IT, REFINE_BETA, REFINE_LR = (3 if QUICK_TEST else 25), 64, 0.02
if REFINE:
    W_CROSS = 3.0
    x = np.load("ringcoupler_design_x.npy")
    m = np.zeros_like(x); v = np.zeros_like(x); st = 0
    for it in range(REFINE_IT):
        f0, g = fom_and_grad(x, REFINE_BETA)
        st += 1
        m = 0.9 * m + 0.1 * g; v = 0.999 * v + 0.001 * g * g
        x = np.clip(x + REFINE_LR * (m / (1 - 0.9 ** st)) / (np.sqrt(v / (1 - 0.999 ** st)) + 1e-8), 0, 1)
        print(f"refine {it + 1:>2}/{REFINE_IT}  J={f0: .5f}  cross={LAST['Pk']:.4f} (target {K_T:.4f})  through={LAST['Pt']:.4f}")
    np.save("ringcoupler_design_x.npy", x)
    rho_final = np.asarray(mapping(x, ETA, 256.0)).reshape(Nx, Ny)
    np.save("ringcoupler_design_rho.npy", rho_final)
    W_CROSS = 1.0

## ۳) راستی‌آزمایی کوپلر با رزولوشن بالاتر

In [ ]:
def run_coupler(rho2d, res, nf):
    geo = stub_blocks() + [mp.Block(center=mp.Vector3(), size=mp.Vector3(DR_X, DR_Y, 0),
                                    material=mp.MaterialGrid(mp.Vector3(Nx, Ny), SiO2, Si2D, weights=rho2d,
                                                             do_averaging=False, grid_type="U_MEAN"))]
    sim = make_csim(geo, res)
    pos = {"thru": (XMO_C, -hs), "cross": (XMO_C, +hs), "refl": (XMI_C, -hs), "back": (XMI_C, +hs)}
    mons = {k: sim.add_mode_monitor(cmid, cspan, nf, mp.ModeRegion(center=mp.Vector3(cx, cy, 0), size=mp.Vector3(0, src_size, 0)))
            for k, (cx, cy) in pos.items()}
    dft = sim.add_dft_fields([mp.Ey], fcen_src, 0, 1, where=mp.Volume(center=mp.Vector3(),
                             size=mp.Vector3(sxc - 2 * dpml, syc - 2 * dpml, 0)))
    sim.run(until_after_sources=T_C)
    al = {k: np.array(sim.get_eigenmode_coefficients(mn, [1], eig_parity=PARITY).alpha[0]) for k, mn in mons.items()}
    return al, np.array(mp.get_flux_freqs(mons["thru"])), np.real(sim.get_dft_array(dft, mp.Ey, 0))


print("Coupler verification ...")
a_ref_v, freqs_c = cref_alpha(RES_VER, NF_CPL)
al, _, fld_c = run_coupler(rho_final, RES_VER, NF_CPL)
t_cpl = al["thru"][:, 0] / a_ref_v
k_cpl = al["cross"][:, 0] / a_ref_v
r_cpl = al["refl"][:, 1] / a_ref_v
b_cpl = al["back"][:, 1] / a_ref_v
lam_c = 1 / freqs_c
o = np.argsort(lam_c)
dB = lambda p: 10 * np.log10(np.maximum(p, 1e-12))
Pk_, Pt_ = np.abs(k_cpl) ** 2, np.abs(t_cpl) ** 2

fig, ax = plt.subplots(1, 3, figsize=(16, 4))
ax[0].plot(lam_c[o], Pk_[o], label="cross  (kappa^2)"); ax[0].plot(lam_c[o], Pt_[o], label="through")
ax[0].axhline(KAPPA2, color="gray", ls="--"); ax[0].set_xlabel("Wavelength (um)"); ax[0].set_ylabel("|S|^2")
ax[0].grid(alpha=0.3); ax[0].legend(); ax[0].set_title(f"target kappa^2 = {KAPPA2:.3f}")
ax[1].plot(lam_c[o], dB(np.abs(r_cpl) ** 2)[o], label="reflection"); ax[1].plot(lam_c[o], dB(np.abs(b_cpl) ** 2)[o], label="backward ring port")
ax[1].set_xlabel("Wavelength (um)"); ax[1].set_ylabel("Power (dB)"); ax[1].set_ylim(-50, 0); ax[1].grid(alpha=0.3); ax[1].legend()
ext = [-(sxc - 2 * dpml) / 2, (sxc - 2 * dpml) / 2, -(syc - 2 * dpml) / 2, (syc - 2 * dpml) / 2]
vm = np.max(np.abs(fld_c))
ax[2].imshow(fld_c.T, origin="lower", extent=ext, cmap="RdBu", vmin=-vm, vmax=vm); ax[2].set_title("Re(Ey) at 1.55 um, input = bus left")
plt.tight_layout(); plt.show()

print(f"kappa^2: mean {Pk_.mean():.4f} | min {Pk_.min():.4f} | max {Pk_.max():.4f}  (target {KAPPA2:.4f})")
print(f"excess loss (band mean): {np.mean(-10 * np.log10(Pk_ + Pt_)):.3f} dB | "
      f"worst reflection: {dB(np.abs(r_cpl) ** 2).max():.1f} dB | worst backward coupling: {dB(np.abs(b_cpl) ** 2).max():.1f} dB")
print(f"phase(k/t): {np.degrees(np.angle(k_cpl / t_cpl)).mean():.1f} deg (ideal: +-90)")
if abs(Pk_.mean() - KAPPA2) / KAPPA2 > 0.3:
    raise RuntimeError(f"Coupler kappa^2 = {Pk_.mean():.4f} is not close to the target {KAPPA2:.4f}. "
                       "Do NOT assemble the ring yet: re-run the optimisation (QUICK_TEST=False, adjust G0).")

## ۴) مونتاژ رینگ مسابقه‌ای (دو کوپلر طراحی‌شده) و FDTD

پایین: کوپلر برای باس ورودی/Through. بالا: همان کوپلر **۱۸۰° چرخیده** برای باس Drop.

In [ ]:
class CL:
    def __init__(self, x, y, th=0.0):
        self.x, self.y, self.th = float(x), float(y), float(th)
        self.p = [np.array([[self.x, self.y]])]

    def straight(self, L):
        if L > 1e-12:
            self.x += L * np.cos(self.th); self.y += L * np.sin(self.th)
            self.p.append(np.array([[self.x, self.y]]))
        return self

    def arc(self, R, ang, n=120):         # ang > 0: left turn
        sg = np.sign(ang)
        cx = self.x - sg * R * np.sin(self.th); cy = self.y + sg * R * np.cos(self.th)
        a0 = np.arctan2(self.y - cy, self.x - cx)
        a = a0 + np.linspace(0, ang, n + 1)[1:]
        pts = np.column_stack((cx + R * np.cos(a), cy + R * np.sin(a)))
        self.p.append(pts)
        self.x, self.y = float(pts[-1, 0]), float(pts[-1, 1]); self.th += ang
        return self

    def pts(self):
        return np.vstack(self.p)


def to_polygon(c, width):
    x, y = c[:, 0], c[:, 1]
    dx, dy = np.gradient(x), np.gradient(y)
    nrm = np.hypot(dx, dy); nx, ny = -dy / nrm, dx / nrm
    return np.vstack((np.column_stack((x + nx * width / 2, y + ny * width / 2)),
                      np.column_stack((x - nx * width / 2, y - ny * width / 2))[::-1]))


def prism(poly):
    return mp.Prism([mp.Vector3(float(a), float(b), -500.0) for a, b in poly], height=1000.0, material=Si2D)


xe, ov = DR_X / 2, 0.3
Y_TOP = 2 * PORT_SEP + 2 * R_LOOP                 # drop-bus y (bus-in is at y = 0)
YC_BOT, YC_TOP = hs, Y_TOP - hs                   # design-region centres
X_ext = xe + L_ST + R_LOOP                        # outermost x of the ring
Xb = X_ext + 3.0                                  # bus half length


def ring_polys():
    right = CL(xe - ov, PORT_SEP).straight(ov + L_ST).arc(R_LOOP, np.pi).straight(L_ST + ov)
    left = CL(-(xe - ov), PORT_SEP + 2 * R_LOOP, th=np.pi).straight(ov + L_ST).arc(R_LOOP, np.pi).straight(L_ST + ov)
    assert abs(right.x - (xe - ov)) < 1e-6 and abs(left.x + (xe - ov)) < 1e-6
    return [to_polygon(right.pts(), w), to_polygon(left.pts(), w)]


def bus_polys(with_top=True):
    out = []
    ys = [0.0] + ([Y_TOP] if with_top else [])
    for yb_ in ys:
        out.append(to_polygon(CL(-Xb, yb_).straight(Xb - (xe - ov)).pts(), w))
        out.append(to_polygon(CL(xe - ov, yb_).straight(Xb - (xe - ov)).pts(), w))
    return out


rho_top = rho_final[::-1, ::-1].copy()            # 180-degree rotation (design is already y-symmetric)


def dr_block(yc, rho2d):
    return mp.Block(center=mp.Vector3(0, yc, 0), size=mp.Vector3(DR_X, DR_Y, 0),
                    material=mp.MaterialGrid(mp.Vector3(Nx, Ny), SiO2, Si2D, weights=rho2d,
                                             do_averaging=False, grid_type="U_MEAN"))


# ---- preview
fig, ax = plt.subplots(figsize=(11, 7))
for p in ring_polys() + bus_polys():
    ax.fill(p[:, 0], p[:, 1], color="C0", alpha=0.55)
for yc, rr in ((YC_BOT, rho_final), (YC_TOP, rho_top)):
    ax.imshow(rr.T, origin="lower", cmap="Blues", extent=[-xe, xe, yc - DR_Y / 2, yc + DR_Y / 2], alpha=0.95)
ax.set_aspect("equal"); ax.set_xlim(-Xb, Xb); ax.set_ylim(-1.5, Y_TOP + 1.5)
ax.set_title(f"Inverse-designed racetrack ring (ring length ~ {L_RING_EST:.0f} um)"); plt.show()

In [ ]:
# ---- full-ring simulation cell
Lxi = X_ext + 2.5
sxr = 2 * (Lxi + dpml)
y_lo, y_hi = -(src_size / 2 + 0.6 + dpml), Y_TOP + src_size / 2 + 0.6 + dpml
syr = y_hi - y_lo
XS_R, XM_R = -(X_ext + 1.6), X_ext + 1.0
DECAY_TOL, T_MIN, T_MAX = 1e-5, 200, (600 if QUICK_TEST else 4000)


def run_ring_full(res, nf, with_ring=True):
    t0 = time.time()
    if with_ring:
        geo = [prism(p) for p in ring_polys() + bus_polys(True)] + [dr_block(YC_BOT, rho_final), dr_block(YC_TOP, rho_top)]
    else:
        geo = [mp.Block(size=mp.Vector3(mp.inf, w, mp.inf), center=mp.Vector3(0, 0, 0), material=Si2D)]
    src = mp.EigenModeSource(mp.GaussianSource(1 / 1.55, fwidth=0.03), center=mp.Vector3(XS_R, 0, 0),
                             size=mp.Vector3(0, src_size, 0), eig_band=1, eig_parity=PARITY, eig_match_freq=True)
    sim = mp.Simulation(cell_size=mp.Vector3(sxr, syr, 0), geometry_center=mp.Vector3(0, 0.5 * (y_lo + y_hi), 0),
                        boundary_layers=[mp.PML(dpml)], geometry=geo, sources=[src], default_material=SiO2,
                        resolution=res, dimensions=2)
    pos = {"thru": (XM_R, 0.0), "drop": (-XM_R, Y_TOP), "refl": (-XM_R, 0.0)} if with_ring else {"thru": (XM_R, 0.0)}
    mons = {k: sim.add_mode_monitor(rmid, rspan, nf, mp.ModeRegion(center=mp.Vector3(cx, cy, 0), size=mp.Vector3(0, src_size, 0)))
            for k, (cx, cy) in pos.items()}
    sim.run(until_after_sources=mp.stop_when_dft_decayed(tol=DECAY_TOL, minimum_run_time=T_MIN, maximum_run_time=T_MAX))
    al_ = {k: np.array(sim.get_eigenmode_coefficients(m_, [1], eig_parity=PARITY).alpha[0]) for k, m_ in mons.items()}
    print(f"   done in {time.time() - t0:.0f} s (sim time {sim.meep_time():.0f})")
    return al_, np.array(mp.get_flux_freqs(mons["thru"]))


print("Reference run (straight bus) ...")
al_r, _ = run_ring_full(RES_VER, NF_RING, with_ring=False)
a_ref_r = al_r["thru"][:, 0]
print("Ring run ...")
al_f, fr = run_ring_full(RES_VER, NF_RING, with_ring=True)
oo = np.argsort(1 / fr); lam = (1 / fr)[oo]
Tt = (np.abs(al_f["thru"][:, 0]) ** 2 / np.abs(a_ref_r) ** 2)[oo]
Td = (np.abs(al_f["drop"][:, 1]) ** 2 / np.abs(a_ref_r) ** 2)[oo]
Rr = (np.abs(al_f["refl"][:, 1]) ** 2 / np.abs(a_ref_r) ** 2)[oo]

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(lam, dB(Tt), label="Through"); ax[0].plot(lam, dB(Td), label="Drop"); ax[0].plot(lam, dB(Rr), label="Reflection")
ax[0].set_ylim(-45, 2); ax[0].set_xlabel("Wavelength (um)"); ax[0].set_ylabel("Power (dB)"); ax[0].grid(alpha=0.3); ax[0].legend()
ax[0].set_title("Inverse-designed-coupler ring: FDTD")
ax[1].plot(lam, Tt, label="Through"); ax[1].plot(lam, Td, label="Drop"); ax[1].plot(lam, Tt + Td, "k:", label="Through + Drop")
ax[1].set_ylim(0, 1.05); ax[1].set_xlabel("Wavelength (um)"); ax[1].set_ylabel("|S|^2"); ax[1].grid(alpha=0.3); ax[1].legend()
plt.tight_layout(); plt.show()

### تشخیص: افت یک نیم‌حلقه (نیم‌دایره + مستقیم‌ها)

افت چرخهٔ فیت‌شده باید فقط مربوط به خم‌ها/مستقیم‌ها باشد (افت کوپلرها جدا در `r` می‌آید).
این سلول افت یک نیم‌دایرهٔ `R_LOOP` با دو بخش مستقیم را مستقیم اندازه می‌گیرد، تا بدانیم افت چرخه از خم است یا نه.

In [ ]:
RUN_HALF_LOOP_DIAG = True


def run_half(geo, mon_xy, idx, res, nf=5):
    xa, xb, ya, yb = -4.0, R_LOOP + 3.5, -2.0, 2 * R_LOOP + 2.0
    src = mp.EigenModeSource(mp.GaussianSource(1 / 1.55, fwidth=0.03), center=mp.Vector3(-1.4, 0, 0),
                             size=mp.Vector3(0, src_size, 0), eig_band=1, eig_parity=PARITY, eig_match_freq=True)
    sim = mp.Simulation(cell_size=mp.Vector3(xb - xa, yb - ya, 0), geometry_center=mp.Vector3(0.5 * (xa + xb), 0.5 * (ya + yb), 0),
                        boundary_layers=[mp.PML(dpml)], geometry=geo, sources=[src], default_material=SiO2,
                        resolution=res, dimensions=2)
    mon = sim.add_mode_monitor(rmid, rspan, nf, mp.ModeRegion(center=mp.Vector3(*mon_xy, 0), size=mp.Vector3(0, src_size, 0)))
    sim.run(until_after_sources=NG_EST * (3 + np.pi * R_LOOP) + 40)
    return np.array(sim.get_eigenmode_coefficients(mon, [1], eig_parity=PARITY).alpha[0])[:, idx]


if RUN_HALF_LOOP_DIAG:
    half = CL(-2.0, 0.0).straight(1.5).arc(R_LOOP, np.pi).straight(1.5)            # ends at (-2.0, 2R) heading -x
    a_h = run_half([prism(to_polygon(half.pts(), w))], (-1.4, 2 * R_LOOP), 1, RES_VER)
    a_s = run_half([prism(to_polygon(CL(-2.0, 0.0).straight(8.0).pts(), w))], (-0.6, 0.0), 0, RES_VER)
    T_half = np.mean(np.abs(a_h) ** 2 / np.abs(a_s) ** 2)
    print(f"half loop (semicircle R = {R_LOOP} um + 2 x 1.5 um straight): loss = {-10 * np.log10(T_half):.3f} dB")
    print(f"=> two half loops: about {-20 * np.log10(T_half) :.2f} dB per round trip from bends/transitions "
          f"(your conventional ring fit: 0.78 dB incl. its couplers)")

## ۵) استخراج Q، FSR، نسبت خاموشی و مقایسه با هدف

In [ ]:
pk, _ = find_peaks(Td, prominence=0.1 * Td.max())
print(f"Found {len(pk)} drop resonances at lambda (um):", np.round(lam[pk], 4))
widths = peak_widths(Td, pk, rel_height=0.5)[0]
dlam = np.mean(np.abs(np.diff(lam)))
for i, p in enumerate(pk):
    fwhm = widths[i] * dlam
    print(f"  lambda_res = {lam[p]:.4f} um | FWHM = {fwhm*1e3:.3f} nm | Q_loaded = {lam[p]/fwhm:,.0f} | drop peak = {dB(Td[p]):.2f} dB")
if len(pk) >= 2:
    print(f"Measured FSR = {np.mean(np.abs(np.diff(lam[pk]))) * 1e3:.2f} nm "
          f"(expected {lam0**2 / (ng_c * L_RING_EST) * 1e3:.2f} nm)")
print(f"Through extinction (max/min in window): {dB(Tt.max()) - dB(Tt.min()):.1f} dB")


def ring_model(f, r, k2, a, FSR_f, f_res):
    # two identical couplers: through amplitude r, cross POWER k2 (r^2 + k2 <= 1), round-trip amplitude a
    phi = 2 * np.pi * (f - f_res) / FSR_f
    l_ = r * r + k2
    D = 1 - 2 * r * r * a * np.cos(phi) + (r * r * a) ** 2
    return r * r * (1 - 2 * l_ * a * np.cos(phi) + (l_ * a) ** 2) / D, k2 * k2 * a / D


if len(pk) == 0:
    raise RuntimeError("No drop resonance found: the coupler probably does not couple (check section 3).")
f_arr = 1 / lam
win = (lam_c >= R_WL_MIN) & (lam_c <= R_WL_MAX)
Pt_win, Pk_win = float(np.mean(Pt_[win])), float(np.mean(Pk_[win]))
p0 = pk[np.argmax(Td[pk])]
FSR0 = 1 / (ng_c * L_RING_EST)
lb = [0.5, 0.005, 0.3, 0.7 * FSR0, f_arr[p0] - 0.3 * FSR0]
ub = [0.9999, 0.6, 1.0, 1.3 * FSR0, f_arr[p0] + 0.3 * FSR0]


def _res(p):
    mt_, md_ = ring_model(f_arr, *p)
    return np.r_[mt_ - Tt, md_ - Td]


fit = least_squares(_res, [np.sqrt(Pt_win), Pk_win, 0.95, FSR0, f_arr[p0]], bounds=(lb, ub))
r_f, k2_f, a_f, FSR_f_f, f_res_f = fit.x
print("\nFit (lossy-coupler add-drop model):")
print(f"  through power r^2 = {r_f**2:.4f}   (coupler FDTD, window mean: {Pt_win:.4f})")
print(f"  cross power kappa^2 = {k2_f:.4f}   (coupler FDTD, window mean: {Pk_win:.4f}; design target {KAPPA2:.4f})")
print(f"  round-trip loss, couplers excluded = {-20 * np.log10(a_f):.3f} dB   (assumed in the design: {A_RT_DB:.2f} dB)")
print(f"  FSR = {(1 / f_res_f) ** 2 * FSR_f_f * 1e3:.2f} nm")

mt, mdp = ring_model(f_arr, *fit.x)
plt.figure(figsize=(8, 4))
plt.plot(lam, Tt, "C0-", label="FDTD through"); plt.plot(lam, Td, "C1-", label="FDTD drop")
plt.plot(lam, mt, "C0--", label="model through"); plt.plot(lam, mdp, "C1--", label="model drop")
plt.xlabel("Wavelength (um)"); plt.ylabel("|S|^2"); plt.grid(alpha=0.3); plt.legend(ncol=2)
plt.title("Inverse-designed ring: FDTD vs fitted model"); plt.show()

### تست سازگاری: فیت با کوپلرِ ثابت

فیت آزاد `r` و `κ²` را هم تغییر می‌دهد و بین `κ²` و `a` تبادل (degeneracy) دارد. اینجا `r²` و `κ²` را روی مقادیر **اندازه‌گیری‌شدهٔ کوپلر** قفل می‌کنیم
و فقط `a`، FSR و فرکانس تشدید فیت می‌شوند. اگر باز هم خوب فیت شد، کوپلر در رینگ همان‌طور رفتار می‌کند که تنهایی؛ اگر نه، چیزی در مونتاژ فرق دارد.

In [ ]:
def _res_fixed(p):
    mt_, md_ = ring_model(f_arr, np.sqrt(Pt_win), Pk_win, *p)
    return np.r_[mt_ - Tt, md_ - Td]


fit_c = least_squares(_res_fixed, [0.8, FSR_f_f, f_res_f],
                      bounds=([0.2, 0.7 * FSR0, f_arr[p0] - 0.3 * FSR0], [1.0, 1.3 * FSR0, f_arr[p0] + 0.3 * FSR0]))
a_c = fit_c.x[0]
rms_free = float(np.sqrt(np.mean(_res(fit.x) ** 2)))
rms_fix = float(np.sqrt(np.mean(_res_fixed(fit_c.x) ** 2)))
mt_c, md_c = ring_model(f_arr, np.sqrt(Pt_win), Pk_win, *fit_c.x)
print(f"free fit      : RMS error = {rms_free:.4f}  | r^2 = {r_f**2:.3f}, kappa^2 = {k2_f:.3f}, round-trip loss = {-20*np.log10(a_f):.2f} dB")
print(f"coupler fixed : RMS error = {rms_fix:.4f}  | r^2 = {Pt_win:.3f}, kappa^2 = {Pk_win:.3f}, round-trip loss = {-20*np.log10(a_c):.2f} dB")
print(f"FDTD          : drop peak = {dB(Td.max()):.2f} dB | through min = {dB(Tt.min()):.2f} dB | through max = {dB(Tt.max()):.2f} dB")
mt_f, md_f = ring_model(f_arr, *fit.x)
print(f"free model    : drop peak = {dB(md_f.max()):.2f} dB | through min = {dB(mt_f.min()):.2f} dB")
print(f"fixed model   : drop peak = {dB(md_c.max()):.2f} dB | through min = {dB(mt_c.min()):.2f} dB")
if rms_fix <= 1.5 * rms_free:
    print("=> consistent: the isolated-coupler numbers explain the ring; the large round-trip loss is real (bends / transitions).")
else:
    print("=> NOT consistent: the ring sees different coupling than the isolated coupler; check the assembly (ports, rotation, overlaps).")

plt.figure(figsize=(8, 4))
plt.plot(lam, Tt, "C0-", label="FDTD through"); plt.plot(lam, Td, "C1-", label="FDTD drop")
plt.plot(lam, mt_c, "C0--", label="model (coupler fixed)"); plt.plot(lam, md_c, "C1--", label="model drop (coupler fixed)")
plt.xlabel("Wavelength (um)"); plt.ylabel("|S|^2"); plt.grid(alpha=0.3); plt.legend(ncol=2); plt.show()

## ۶) ذخیره‌سازی و GDS

In [ ]:
np.savez("invring_results.npz", wavelength_um=lam, through=Tt, drop=Td, reflection=Rr, kappa2_target=KAPPA2,
         coupler_wavelength_um=lam_c, coupler_t=t_cpl, coupler_k=k_cpl, rho=rho_final)
try:
    import gdstk
    from scipy.ndimage import zoom

    def design_to_polys(rho2d, x_left, y_center, up=4):
        big = zoom(rho2d, up, order=1) > 0.5
        px, py = DR_X / big.shape[0], DR_Y / big.shape[1]
        rects = []
        for iy in range(big.shape[1]):
            d_ = np.diff(np.concatenate([[0], big[:, iy].astype(np.int8), [0]]))
            for s_, e_ in zip(np.where(d_ == 1)[0], np.where(d_ == -1)[0]):
                rects.append(gdstk.rectangle((x_left + s_ * px, y_center - DR_Y / 2 + iy * py),
                                             (x_left + e_ * px, y_center - DR_Y / 2 + (iy + 1) * py)))
        return rects

    shapes = [gdstk.Polygon([tuple(q) for q in p], layer=2, datatype=0) for p in ring_polys() + bus_polys(True)]
    shapes += design_to_polys(rho_final, -xe, YC_BOT) + design_to_polys(rho_top, -xe, YC_TOP)
    merged = gdstk.boolean(shapes, [], "or", precision=1e-3, layer=2, datatype=0)
    lib = gdstk.Library(); cg = lib.new_cell("INVDES_RING")
    for p_ in merged:
        cg.add(p_)
    lib.write_gds("invdes_ring.gds")
    print("Saved invring_results.npz, ringcoupler_design_rho.npy, invdes_ring.gds")
except Exception as e:
    print("GDS export skipped:", e)